# 09 — Quantization Comparison (FP32 vs FP16 vs INT8)

So sánh accuracy (Precision/Recall/mAP50/mAP50-95) và kích thước model giữa 3 mức lượng tử hóa,
trên cùng bộ 100 ảnh benchmark (`AI/benchmark/pc_subset_dataset/`), cho 2 model đại diện
`yolo11n_640` và `yolo11s_640`.

**Lưu ý về phương pháp đo (quan trọng):**
- Số liệu **FP16/INT8** trong notebook này được đo bằng `ultralytics.YOLO(...).val()` — pipeline
  validate chính thức của ultralytics, chạy trực tiếp trên notebook 08.
- Số liệu **FP32 baseline** lấy từ `AI/benchmark/pc_100image_4model_metrics.csv` (đã có sẵn trong
  repo từ trước, đo bằng script PC-side riêng của nhóm — không phải `ultralytics.val()`).
- Lý do không dùng chung một pipeline: model FP32 gốc export theo layout **NCHW** (`ai-edge-torch`),
  trong khi `ultralytics.YOLO(...).val()` cho TFLite chỉ hỗ trợ layout **NHWC** — chạy thử trực tiếp
  báo lỗi `Dimension mismatch. Got 640 but expected 3`. Viết lại một validator NCHW tương thích
  ultralytics không đáng thời gian bỏ ra so với lợi ích, nên notebook này **giữ nguyên số liệu FP32
  đã có** và ghi rõ nguồn khác nhau thay vì cố ép về cùng 1 script.
- Vì vậy: so sánh **FP16 vs INT8 với nhau** là hoàn toàn đồng nhất (cùng script, cùng bộ ảnh).
  So sánh **FP32 vs FP16/INT8** vẫn cho biết xu hướng đúng (quantization giữ được phần lớn accuracy,
  size giảm mạnh) nhưng chênh lệch tuyệt đối giữa FP32 và FP16/INT8 nên được diễn giải thận trọng
  vì khác pipeline đo.


In [ ]:
import pandas as pd

# FP32 baseline — copy từ AI/benchmark/pc_100image_4model_metrics.csv (đo bằng script PC-side của nhóm)
fp32 = {
    "yolo11n_640": {"precision": 0.780523, "recall": 0.660411, "mAP50": 0.729657, "mAP50_95": 0.577848, "size_mib": 10.178},
    "yolo11s_640": {"precision": 0.935172, "recall": 0.942058, "mAP50": 0.970428, "mAP50_95": 0.806390, "size_mib": 36.278},
}

# FP16 / INT8 — đo trong notebook 08 bằng ultralytics.YOLO(...).val() trên cùng bộ 100 ảnh
fp16 = {
    "yolo11n_640": {"precision": 0.780068, "recall": 0.834071, "mAP50": 0.896638, "mAP50_95": 0.756677, "size_mib": 5.140},
    "yolo11s_640": {"precision": 0.898972, "recall": 0.877625, "mAP50": 0.932896, "mAP50_95": 0.799555, "size_mib": 18.191},
}

int8 = {
    "yolo11n_640": {"precision": 0.801421, "recall": 0.860398, "mAP50": 0.902174, "mAP50_95": 0.755407, "size_mib": 2.861},
    "yolo11s_640": {"precision": 0.906738, "recall": 0.877792, "mAP50": 0.932389, "mAP50_95": 0.795879, "size_mib": 9.488},
}

rows = []
for model_name in ["yolo11n_640", "yolo11s_640"]:
    for precision_name, data in [("FP32", fp32), ("FP16", fp16), ("INT8", int8)]:
        row = {"model": model_name, "precision_mode": precision_name}
        row.update(data[model_name])
        rows.append(row)

df = pd.DataFrame(rows)
display(df.round(4))


## 1. Model size vs precision

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(8, 5))
for model_name in ["yolo11n_640", "yolo11s_640"]:
    sub = df[df["model"] == model_name].set_index("precision_mode").loc[["FP32", "FP16", "INT8"]]
    ax.plot(sub.index, sub["size_mib"], marker="o", label=model_name)

ax.set_ylabel("Model size (MiB)")
ax.set_title("Kích thước model theo mức lượng tử hóa")
ax.legend()
plt.tight_layout()
plt.show()


## 2. mAP50-95 vs precision

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for model_name in ["yolo11n_640", "yolo11s_640"]:
    sub = df[df["model"] == model_name].set_index("precision_mode").loc[["FP32", "FP16", "INT8"]]
    ax.plot(sub.index, sub["mAP50_95"], marker="o", label=model_name)

ax.set_ylabel("mAP50-95")
ax.set_title("Accuracy (mAP50-95) theo mức lượng tử hóa")
ax.set_ylim(0, 1.0)
ax.legend()
plt.tight_layout()
plt.show()


## 3. Trade-off: size reduction vs accuracy retained

In [ ]:
summary_rows = []
for model_name in ["yolo11n_640", "yolo11s_640"]:
    base = df[(df["model"] == model_name) & (df["precision_mode"] == "FP32")].iloc[0]
    for mode in ["FP16", "INT8"]:
        row = df[(df["model"] == model_name) & (df["precision_mode"] == mode)].iloc[0]
        summary_rows.append({
            "model": model_name,
            "precision_mode": mode,
            "size_reduction_percent": (1 - row["size_mib"] / base["size_mib"]) * 100,
            "mAP50_95_retained_percent": row["mAP50_95"] / base["mAP50_95"] * 100,
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df.round(2))


## Nhận xét

- **INT8** giảm kích thước mạnh nhất (n640: -71.9%, s640: -73.9% so với FP32) trong khi vẫn giữ
  được recall/mAP50 cao hơn hẳn baseline FP32 đo trên cùng bộ 100 ảnh (lưu ý khác pipeline đo như
  đã nêu ở trên) — cho thấy quantization không làm suy giảm chất lượng phát hiện nghiêm trọng.
- **FP16** giảm kích thước ~50% so với FP32, mAP50-95 giữa FP16 và INT8 gần như tương đương nhau
  trên cả 2 model — cả hai đều là ứng viên khả thi để triển khai Android.
- Vì FP16/INT8 đo cùng 1 pipeline (`ultralytics.val()`) trên cùng bộ ảnh, chênh lệch giữa hai mức
  này là **đáng tin cậy trực tiếp**: INT8 nhỉnh hơn FP16 một chút ở n640, gần như bằng nhau ở s640.
- **Quyết định cuối cùng** giữa FP16 và INT8 (và giữa n640/s640) cần thêm trục **latency/FPS/RAM
  trên thiết bị Android thật** — xem Phase 2 (GPU/NNAPI delegate sweep) và notebook
  `10_final_tradeoff.ipynb` sau khi có số liệu benchmark thật.
